# Section 1: Ensemble data assimilation concepts in 1D

*(Replaces DART_LAB slide deck Section 1.)*

Data assimilation combines a **forecast** of a system's state (the *prior*)
with **observations** to produce a better estimate (the *posterior*). This
notebook builds the machinery one piece at a time, for a single scalar
variable: Bayes' rule, the product of Gaussians, the Kalman filter, and
finally ensemble filters.

In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import pydartlab as dl
import pydartlab.apps as apps

## Bayes' rule

For a state $T$ (say, an unknown temperature) and an observation $T_o$:

$$ p(T \mid T_o) = \frac{p(T_o \mid T)\, p(T)}{p(T_o)} $$

* $p(T)$ — the **prior**: what we believe before seeing the observation,
* $p(T_o \mid T)$ — the **likelihood**: how probable the observed value is
  if the truth were $T$ (set by the instrument's error distribution),
* $p(T \mid T_o)$ — the **posterior**.

The denominator is just the normalization. When the prior is
$N(\bar{T}_p, \sigma_p^2)$ and the observation error is
$N(0, \sigma_o^2)$, the posterior is **also Gaussian** with

$$ \sigma_a^2 = \left( \sigma_p^{-2} + \sigma_o^{-2} \right)^{-1}, \qquad
   \bar{T}_a = \sigma_a^2 \left( \sigma_p^{-2} \bar{T}_p + \sigma_o^{-2} T_o \right). $$

## Exercise: `gaussian_product`

Run the tool and explore:

1. Change the **prior mean** and **observation**: the posterior mean always
   lies *between* them.
2. Change the **standard deviations**: the posterior SD is always *smaller*
   than both — combining information always reduces uncertainty.
3. Make the prior much more uncertain than the observation (and vice
   versa): the posterior hugs whichever is more certain.

In [ ]:
gp = apps.gaussian_product()
gp

In [ ]:
# Mouse-free version of the exercise:
for prior in [(0, 1), (0, 4)]:
    gp.prior_mean.value, gp.prior_sd.value = prior
    gp.obs_mean.value, gp.obs_sd.value = 2.0, 1.0
    gp.update()
    print(f"prior {prior} -> posterior mean/sd/weight: "
          f"{np.round(gp.last_result, 3)}")

## Cycling: assimilate, forecast, repeat

Real assimilation is *cycled*: assimilate an observation, advance the model
to the next observation time, and repeat. With a linear forecast model

$$ x_{t+1} = G\, x_t $$

growth rate $G > 1$ amplifies forecast errors between observation times,
so the filter has to keep working. With $G = 1$ each new observation just
shrinks the uncertainty further. This is the **Kalman filter** in one
dimension.

## Exercise: `oned_cycle` (continuous Kalman filter)

Create an ensemble (click in the lower panel, then click outside it), then
press **Cycle DA** repeatedly:

1. With growth rate 1: watch the prior/posterior SD shrink every cycle.
2. Set growth rate 2: the SD now reaches a steady state — error growth from
   the model balances the information from each observation.
3. Change the observation error SD and find how the steady state responds.

In [ ]:
oc = apps.oned_cycle()
oc

In [ ]:
# Mouse-free: same experiment, scripted
oc.growth_rate.value = 2.0
oc.set_ensemble([-1.0, 0.5, 1.0, 2.0, 3.5])
for _ in range(8):
    oc.cycle()
print("KF sd after 8 cycles with G=2:", round(oc.experiment.kf_sd, 3))

## Ensembles

Real models are too big (and too nonlinear) for the continuous Kalman
filter. **Ensemble** filters represent the prior with a sample of $N$ model
states. Three ways to use an observation to update an ensemble:

* **EAKF** (Ensemble Adjustment Kalman Filter): fit a Gaussian to the
  ensemble, compute the Gaussian posterior, then *shift and linearly
  contract* the members to match it. Deterministic; preserves the ensemble
  shape.
* **EnKF** (perturbed-observations): pair each member with a randomly
  perturbed copy of the observation. Stochastic.
* **RHF** (Rank Histogram Filter): represent the prior with a "rank
  histogram" — $1/(N{+}1)$ probability between each pair of sorted members
  with Gaussian tails — multiply by the likelihood, and place the posterior
  members at constant quantiles. Handles non-Gaussian priors
  (more in Section 4).

## Exercise: `oned_ensemble`

Create ensembles (fewer than 10 members works best) and **Update**:

1. A nearly uniformly spaced ensemble — compare the three filters.
2. An ensemble entirely on one side of the observation.
3. A *bimodal* ensemble (two clusters): EAKF/EnKF treat it as one wide
   Gaussian; RHF respects the clusters.
4. Add a single distant outlier: how does each filter move it?

In [ ]:
oe = apps.oned_ensemble()
oe

In [ ]:
# Mouse-free, e.g. the bimodal case with RHF:
oe.filter_radio.value = "RHF"
oe.set_ensemble([-2.2, -1.8, -2.0, 1.8, 2.0, 2.2])
oe.update_ensemble()

## Exercise: a complete cycling system — `oned_model`

`oned_model` runs the full cycle automatically. The truth is always 0; the
model doubles the state each step, so errors grow between assimilation
times. Observations have error SD 1.

Press **Advance Model / Assimilate Obs** a few times to see one cycle at a
time, then use the auto-run (play) button. Watch:

* the *sawtooth* in the error/spread panel — error grows during forecasts,
  drops at assimilation,
* the rank histograms accumulating (more on these in Section 3),
* the kurtosis trace (EAKF keeps it constant; EnKF and RHF don't).

Try ensemble sizes from 2 to 10: how do error and spread change?

In [ ]:
om = apps.oned_model()
om

In [ ]:
# Scripted equivalent: run 200 cycles for three ensemble sizes
from pydartlab.experiments import OneDExperiment

fig, ax = plt.subplots(figsize=(7, 3.2))
for n in (3, 5, 10):
    exp = OneDExperiment(ens_size=n, filter_type="EAKF", seed=42)
    for _ in range(200):
        exp.step()
    err = np.array(exp.history["post_error"])
    ax.plot(np.convolve(err, np.ones(20) / 20, mode="valid"),
            label=f"N = {n} (mean {err.mean():.2f})")
ax.set_xlabel("cycle"); ax.set_ylabel("posterior error (running mean)")
ax.legend(); ax.set_title("Bigger ensembles help, with diminishing returns");

## What you should have seen

* The posterior always lies between prior and observation, and is more
  certain than either.
* Cycling with a growing model reaches a steady-state uncertainty.
* EAKF is a deterministic shift-and-contract; EnKF is its stochastic
  cousin; RHF can represent non-Gaussian priors.
* Small ensembles work surprisingly well in 1D — but error decreases with
  ensemble size.

**Next: Section 2 — what happens when we observe one variable but want to
update others?**